# Machine Learning Pipelines

⚠️ Under construction / not finalized. ⚠️

- Writing and maintaining feature extraction code is complicated
- Well tested ML libraries like scikit-learn are useful for avoiding bugs
- All parameters of feature extraction methods are **hyperparameters** (like the k in KNN)
- Tuning these hyperparameters is important, but complex to implement
- Scikit Learn Pipelines can help with that, too

![ml-pipeline-2.png](figures/ml-pipeline.png)

# Scikit Learn API

[Scikit Learn](https://scikit-learn.org) is a popular machine learning library. 


Its API was copied in many other libraries (e.g. spark.ml). 


It offers most feature transformations, ML models, metrics and many useful tools. 


## Estimators

- implement ``set_params()`` and ``get_params()``
- implement ``fit`` method
- Examples for what ``fit`` does: 
    - [sklearn.feature_extraction.text.CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html): learns dictionary
    - [sklearn.linear_model.Perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html): learns weight vector
    and Pipelines

- implement ``transform`` method
- Examples for what ``transform`` does: 
    - [sklearn.feature_extraction.text.CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html): extracts n-gram counts
    - [sklearn.linear_model.Perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html): projects data onto weight vector

# Pipeline Example: Spam Classifier

![ml-pipeline-2.png](figures/ml-pipeline-2.png)

## Pipelines

Estimators with standardized API can be chained in a Pipeline. 

This allows to chain all feature extraction and classification into one object

Pipelines behave like Estimators themselves, with ``fit`` and ``transform`` methods

All parameters of a Pipeline can be optimized jointly

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import NearestCentroid

reviews = [ 
        "This is a horrible movie", 
        "a great movie",
        "a fantastic movie",
    ]
sentiment = [-1, 1, 1]

text_clf = Pipeline([('vect', CountVectorizer()),
                     ('clf', NearestCentroid())])

text_clf.fit(reviews, sentiment)

text_clf.predict(reviews)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


array([-1,  1,  1])

In [2]:
text_clf.steps

[('vect', CountVectorizer()), ('clf', NearestCentroid())]

In [3]:
text_clf.steps[0][1].transform(reviews)

<3x6 sparse matrix of type '<class 'numpy.int64'>'
	with 8 stored elements in Compressed Sparse Row format>

In [4]:
text_clf.steps[0][1].transform(reviews).toarray()

array([[0, 0, 1, 1, 1, 1],
       [0, 1, 0, 0, 1, 0],
       [1, 0, 0, 0, 1, 0]])

In [5]:
x = text_clf.steps[0][1].transform(reviews)
text_clf.steps[1][1].predict(x)

array([-1,  1,  1])

# Pipelines With Heterogeneous Features

- For best maintainability and clarity ML Pipelines contain **all feature preprocessing steps**

- The ColumnTransformer extracts features from pandas DataFrames into sklearn pipelines

- This allows to optimize **all hyperparameters** end-to-end

- We look at an example from the [sklearn documentation](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)

In [7]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
import pandas as pd

np.random.seed(0)

# Load data from https://www.openml.org/d/40945
X_df, y_series = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True, parser="auto")
X_df.head()

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [8]:
y_series.head()

0    1
1    1
2    0
3    0
4    0
Name: survived, dtype: category
Categories (2, object): ['0', '1']

### Numeric Features

* age: float.
* fare: float.

In [9]:
# We create the preprocessing pipelines for both numeric and categorical data.
numeric_features = ['age', 'fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

### Categoric Features

* embarked: categories encoded as strings {'C', 'S', 'Q'}.
* sex: categories encoded as strings {'female', 'male'}.
* pclass: ordinal integers {1, 2, 3}.

In [10]:
categorical_features = ['embarked', 'sex', 'pclass']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

### Constructing the Pipeline

* ColumnTransformer extracts features and concatenates them

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', LogisticRegression())])

### Training the Pipeline

* ColumnTransformer extracts features and concatenates them

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2)

clf.fit(X_train, y_train)
print("model score: %.3f" % clf.score(X_test, y_test))

model score: 0.790


In [22]:
print(classification_report(y_test, clf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.80      0.88      0.84       162
           1       0.76      0.65      0.70       100

    accuracy                           0.79       262
   macro avg       0.78      0.76      0.77       262
weighted avg       0.79      0.79      0.79       262

